# Fake News Detection — Notebook 1: Data Exploration & Preprocessing

**Project:** A Comparative Study of Classical and Deep Learning Approaches for Fake News Detection

**This notebook covers:**
1. Loading the dataset (Bisaillon's Fake and Real News Dataset)
2. Exploratory data analysis (class balance, text length, vocabulary)
3. Text preprocessing (cleaning, tokenization, stopword removal, stemming)
4. TF-IDF vectorization
5. Train/test split (saved to disk for reuse in Notebooks 2 and 3)

**Output artifacts** (saved to `/kaggle/working/` so they persist between notebook sessions):
- `cleaned_data.csv` — full cleaned dataset
- `train_test_split.npz` — pre-split training and test indices
- `tfidf_vectorizer.pkl` — fitted TF-IDF vectorizer
- `X_train_tfidf.npz`, `X_test_tfidf.npz` — TF-IDF feature matrices
- `y_train.npy`, `y_test.npy` — labels

## 1. Imports and Setup

In [ ]:
import os
import re
import string
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import save_npz, load_npz

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Ensure output directory exists
os.makedirs('/kaggle/working/', exist_ok=True)

print('Setup complete.')

## 2. Loading the Data

The dataset consists of two CSV files:
- `True.csv` — real news articles from Reuters
- `Fake.csv` — fake news articles flagged by PolitiFact and Wikipedia

We download the dataset programmatically using the Kaggle API. **On Kaggle's hosted notebooks, credentials are pre-configured** — no `kaggle.json` setup needed. The dataset gets downloaded into `/kaggle/working/dataset/` on first run.

If you're running this outside Kaggle (e.g., on Google Colab or locally), you'll need to set up the Kaggle API credentials first. See https://www.kaggle.com/docs/api for instructions.

In [ ]:
# Download the dataset using the Kaggle API
import os
import subprocess

DATA_DIR = '/kaggle/working/dataset/'
os.makedirs(DATA_DIR, exist_ok=True)

# Skip download if files already exist (saves time on notebook re-runs)
if not (os.path.exists(os.path.join(DATA_DIR, 'True.csv')) and 
        os.path.exists(os.path.join(DATA_DIR, 'Fake.csv'))):
    print('Downloading dataset from Kaggle...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'clmentbisaillon/fake-and-real-news-dataset',
         '-p', DATA_DIR, '--unzip'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Download failed. STDERR:')
        print(result.stderr)
        print('\nIf the download failed, you can attach the dataset manually:')
        print('  1. Click "+ Add Input" in the right sidebar')
        print('  2. Search for "fake and real news dataset" by Clement Bisaillon')
        print('  3. Set DATA_DIR = "/kaggle/input/fake-and-real-news-dataset/"')
        raise RuntimeError('Dataset download failed')
    print('Download complete.')
else:
    print('Dataset already present, skipping download.')

# Confirm files are there
print(f'\nFiles in {DATA_DIR}:')
for f in os.listdir(DATA_DIR):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Now load the CSVs
true_df = pd.read_csv(os.path.join(DATA_DIR, 'True.csv'))
fake_df = pd.read_csv(os.path.join(DATA_DIR, 'Fake.csv'))

print(f'\nReal news articles: {len(true_df):,}')
print(f'Fake news articles: {len(fake_df):,}')
print(f'Total articles:     {len(true_df) + len(fake_df):,}')

In [ ]:
# Inspect a real news article
print('=== REAL NEWS SAMPLE ===')
print(f"Title:   {true_df.iloc[0]['title']}")
print(f"Subject: {true_df.iloc[0]['subject']}")
print(f"Date:    {true_df.iloc[0]['date']}")
print(f"Text (first 400 chars):\n{true_df.iloc[0]['text'][:400]}")

In [ ]:
# Inspect a fake news article
print('=== FAKE NEWS SAMPLE ===')
print(f"Title:   {fake_df.iloc[0]['title']}")
print(f"Subject: {fake_df.iloc[0]['subject']}")
print(f"Date:    {fake_df.iloc[0]['date']}")
print(f"Text (first 400 chars):\n{fake_df.iloc[0]['text'][:400]}")

## 3. Combining the Datasets and Adding Labels

We add a binary `label` column where `1 = real`, `0 = fake`, then concatenate and shuffle the two dataframes. We use a fixed `random_state=42` throughout for reproducibility.

In [ ]:
true_df['label'] = 1
fake_df['label'] = 0

df = pd.concat([true_df, fake_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Combined dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df["label"].value_counts().rename({0: "Fake", 1: "Real"}))
print(f'\nClass balance: {df["label"].mean():.3f} real / {1 - df["label"].mean():.3f} fake')

## 4. Exploratory Data Analysis

### 4.1 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['label'].value_counts().rename({0: 'Fake', 1: 'Real'})
ax.bar(counts.index, counts.values, color=['#d62728', '#2ca02c'], alpha=0.8)
ax.set_ylabel('Number of articles')
ax.set_title('Class Distribution: Real vs. Fake News')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Subject Distribution

The `subject` column reveals which topics each class covers. This is worth checking because **topic overlap matters**: if real and fake articles cover entirely disjoint subjects, the model could trivially classify based on subject matter rather than learning genuine fake-news indicators.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, title, color in [(axes[0], 1, 'Real News', '#2ca02c'), 
                                  (axes[1], 0, 'Fake News', '#d62728')]:
    subj_counts = df[df['label'] == label]['subject'].value_counts()
    ax.barh(subj_counts.index, subj_counts.values, color=color, alpha=0.8)
    ax.set_title(f'{title} — Subject Distribution')
    ax.set_xlabel('Number of articles')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

print('NOTE: If subjects are entirely disjoint between classes, the model may be learning')
print('a topic classifier rather than a fake-news classifier. We will discuss this in the report.')

### 4.3 Article Length Distribution

We compare article lengths (in words) between classes. Differences in length distribution can be a discriminative signal — fake news is often either very short (clickbait) or padded.

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['label'] == 1]['word_count'], bins=80, alpha=0.5, label='Real', color='#2ca02c', range=(0, 2000))
ax.hist(df[df['label'] == 0]['word_count'], bins=80, alpha=0.5, label='Fake', color='#d62728', range=(0, 2000))
ax.set_xlabel('Word count')
ax.set_ylabel('Number of articles')
ax.set_title('Article Length Distribution by Class (truncated at 2000 words)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Real news word count -- median: {df[df['label']==1]['word_count'].median():.0f}, mean: {df[df['label']==1]['word_count'].mean():.0f}")
print(f"Fake news word count -- median: {df[df['label']==0]['word_count'].median():.0f}, mean: {df[df['label']==0]['word_count'].mean():.0f}")

### 4.4 Most Frequent Words Per Class (Before Cleaning)

A quick look at raw word frequency. This will be dominated by stopwords initially — that's expected and motivates the cleaning step in the next section.

In [ ]:
def top_words(texts, n=15):
    all_words = ' '.join(texts.astype(str)).lower().split()
    return Counter(all_words).most_common(n)

real_top = top_words(df[df['label'] == 1]['text'])
fake_top = top_words(df[df['label'] == 0]['text'])

comparison = pd.DataFrame({
    'Real - word': [w for w, _ in real_top],
    'Real - count': [c for _, c in real_top],
    'Fake - word': [w for w, _ in fake_top],
    'Fake - count': [c for _, c in fake_top],
})

print('Top 15 most frequent words (raw, before cleaning):')
print(comparison.to_string(index=False))

## 5. Text Preprocessing

We apply a four-step cleaning pipeline:

1. **Lowercase** all text
2. **Remove URLs, mentions, special characters, digits** — keep only letters and spaces
3. **Remove English stopwords** (the, of, and, ...) using NLTK's English stopword list
4. **Apply Porter stemming** to collapse morphological variants (running → run, votes → vote)

This pipeline is identical in spirit to what was used in similar text-classification projects, with one project-specific addition: we remove the `"reuters"` token from the cleaned text. Real articles in this dataset systematically begin with `"WASHINGTON (Reuters) -"` style prefixes that would let any model achieve trivial high accuracy by simply detecting that one word. We strip it to ensure the classifier learns more transferable features.

We will join the `title` and `text` fields together and clean them as a single string.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Tokens we want to remove because they leak the source rather than indicate fake-vs-real
leak_tokens = {'reuters', 'reuter'}

def clean_text(text):
    """Full preprocessing pipeline applied to a single string."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)        # URLs
    text = re.sub(r'@\w+', ' ', text)                     # @mentions
    text = re.sub(r'\[.*?\]', ' ', text)                  # bracketed text e.g. [VIDEO]
    text = re.sub(r'<.*?>', ' ', text)                    # HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)                 # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    
    # Tokenize, drop stopwords + leak tokens, stem
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens 
              if t not in stop_words and t not in leak_tokens and len(t) > 1]
    return ' '.join(tokens)

# Test on one article first
print('--- Before cleaning ---')
print(df.iloc[0]['text'][:300])
print('\n--- After cleaning ---')
print(clean_text(df.iloc[0]['text'])[:300])

In [ ]:
# Combine title + text, then clean
# Cleaning ~45k articles takes ~2-3 minutes
print('Cleaning all articles... (this takes a couple of minutes)')
df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['cleaned'] = df['combined'].apply(clean_text)

# Drop empty rows (very rare, but possible if an article was all stopwords/digits)
df = df[df['cleaned'].str.len() > 0].reset_index(drop=True)
print(f'Done. Rows after dropping empty cleaned text: {len(df):,}')

In [ ]:
# Sanity check: top words per class AFTER cleaning
real_top = top_words(df[df['label'] == 1]['cleaned'])
fake_top = top_words(df[df['label'] == 0]['cleaned'])

comparison = pd.DataFrame({
    'Real - word': [w for w, _ in real_top],
    'Real - count': [c for _, c in real_top],
    'Fake - word': [w for w, _ in fake_top],
    'Fake - count': [c for _, c in fake_top],
})

print('Top 15 words AFTER cleaning (stems shown):')
print(comparison.to_string(index=False))

## 6. TF-IDF Vectorization

We use `TfidfVectorizer` with the following choices:

- **`max_features=20000`** — keep the 20,000 most frequent terms. This caps memory and reduces sparsity without sacrificing meaningful vocabulary on a 45k-document corpus.
- **`ngram_range=(1, 2)`** — include both unigrams and bigrams. Bigrams help capture phrases like `donald trump` or `breaking news` that carry meaning beyond their constituent words.
- **`min_df=5`** — drop terms appearing in fewer than 5 documents (typos, rare proper nouns).
- **`max_df=0.7`** — drop terms appearing in more than 70% of documents (corpus-level stopwords that survived our pipeline).
- **`sublinear_tf=True`** — apply log-scaling to term frequency, which is empirically beneficial for long documents.

## 7. Train/Test Split and Vectorization

**Important:** we fit the vectorizer **only on training data** to avoid test-set leakage through the IDF statistics. The fitted vectorizer is then applied to both train and test.

In [ ]:
# 80/20 stratified split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'].values,
    random_state=42,
)

print(f'Training set: {len(X_train_text):,} articles')
print(f'Test set:     {len(X_test_text):,} articles')
print(f'Train class balance: {y_train.mean():.3f}')
print(f'Test class balance:  {y_test.mean():.3f}')

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    sublinear_tf=True,
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f'TF-IDF training matrix shape: {X_train_tfidf.shape}')
print(f'TF-IDF test matrix shape:     {X_test_tfidf.shape}')
print(f'Density: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100:.3f}%')

## 8. Persisting Artifacts to Disk

We save the cleaned dataframe, the fitted vectorizer, the TF-IDF matrices, the labels, and the raw cleaned text strings (for the LSTM in Notebook 3, which needs raw tokens, not TF-IDF).

In [ ]:
OUT_DIR = '/kaggle/working/'

# Save cleaned dataframe (so we don't have to re-clean if we restart)
df[['title', 'text', 'subject', 'cleaned', 'label']].to_csv(
    os.path.join(OUT_DIR, 'cleaned_data.csv'), index=False
)

# Save TF-IDF matrices
save_npz(os.path.join(OUT_DIR, 'X_train_tfidf.npz'), X_train_tfidf)
save_npz(os.path.join(OUT_DIR, 'X_test_tfidf.npz'), X_test_tfidf)

# Save labels
np.save(os.path.join(OUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUT_DIR, 'y_test.npy'), y_test)

# Save raw cleaned text strings (for LSTM tokenizer in Notebook 3)
np.save(os.path.join(OUT_DIR, 'X_train_text.npy'), X_train_text)
np.save(os.path.join(OUT_DIR, 'X_test_text.npy'), X_test_text)

# Save the fitted TF-IDF vectorizer
with open(os.path.join(OUT_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)

print('All artifacts saved to /kaggle/working/:')
for fname in os.listdir(OUT_DIR):
    fpath = os.path.join(OUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:30s}  {size_mb:.2f} MB')

## 9. Summary

**What we built in this notebook:**

1. Loaded a 45k-article fake-vs-real news dataset, class-balanced.
2. Performed EDA on class distribution, subject distribution, article length, and word frequency.
3. Cleaned the text: lowercased, stripped URLs/mentions/non-alphabetic characters, removed English stopwords plus the leak token `reuters`, applied Porter stemming.
4. Performed an 80/20 stratified train/test split.
5. Fitted a TF-IDF vectorizer (unigrams + bigrams, 20k features) **on training data only**.
6. Saved all artifacts to `/kaggle/working/` for use in subsequent notebooks.

**Next:** Notebook 2 — train and evaluate four classical models (Multinomial Naive Bayes, Logistic Regression, Linear SVM, XGBoost), with cross-validated hyperparameter tuning and multi-metric evaluation.